# Pipeline ML — Desbalanceamento Extremo (PRAUC)

Baseado em `hday05_c15_Late_InStyle.ipynb`.

**Cenário:** ~99,995% classe majoritária / ~0,005% minoritária.

**Métrica principal:** PRAUC (`average_precision_score`).

| Fase | Entrada |
|------|---------|
| Treino | `X_train`, `X_test`, `y_train`, `y_test` |
| Produção | `X` (+ `y` opcional para validação) |

## 0. Imports & Config

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import pickle
from pathlib import Path
from datetime import datetime as dt
from time import time

import numpy as np
import pandas as pd

from sklearn import model_selection as ms
from sklearn import ensemble as en
from sklearn import linear_model as lm
from sklearn import neighbors as nh
from sklearn.svm import SVC, NuSVC, LinearSVC
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, classification_report, make_scorer
)
from sklearn.dummy import DummyClassifier

from imblearn.ensemble import BalancedRandomForestClassifier, BalancedBaggingClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from skopt import BayesSearchCV
import optuna

# --- config ---
RANDOM_STATE = 42
MODELS_DIR   = Path('./models')
MODELS_DIR.mkdir(exist_ok=True)

PRIMARY_METRIC = 'PRAUC'  # métrica de ranking / seleção de modelo

## 1. Métricas (PRAUC como principal)

> Usa `predict_proba` sempre que possível — essencial para dados desbalanceados.

In [ ]:
def _get_proba(model, X):
    """Retorna probabilidade da classe positiva (1)."""
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
        return proba[:, 1] if proba.ndim > 1 else proba
    if hasattr(model, 'decision_function'):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    return model.predict(X).astype(float)


def metrics_from_ml_loop(y_true, y_pred, y_proba=None, threshold=0.5):
    """
    Calcula métricas para classificação binária.
    PRAUC é a primeira métrica retornada (principal).
    """
    if y_proba is None:
        y_proba = y_pred.astype(float)

    prauc = average_precision_score(y_true, y_proba)

    return {
        'PRAUC'      : prauc,
        'ROC-AUC'    : roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        'Precision'  : precision_score(y_true, y_pred, zero_division=0),
        'Recall'     : recall_score(y_true, y_pred, zero_division=0),
        'F1'         : f1_score(y_true, y_pred, zero_division=0),
        'Accuracy'   : accuracy_score(y_true, y_pred),
    }


PRAUC_SCORER = make_scorer(average_precision_score, needs_proba=True)

## 2. Tratamento de Desbalanceamento Extremo

Estratégias aplicadas:
1. **`scale_pos_weight`** nos boosters (peso ≈ neg/pos)
2. **`auto_class_weights='Balanced'`** no CatBoost
3. **Modelos nativamente balanceados** (BalancedRandomForest)
4. **Resampling opcional** — undersampling + SMOTE (só se houver amostras minoritárias suficientes)
5. **Threshold tuning** na validação (maximiza F1 ou Recall com precision mínima)
6. **StratifiedKFold** no CV — nunca split aleatório simples

In [ ]:
def imbalance_report(y):
    """Diagnóstico rápido do desbalanceamento."""
    y = np.asarray(y)
    n_pos = (y == 1).sum()
    n_neg = (y == 0).sum()
    total = len(y)
    ratio = n_neg / max(n_pos, 1)

    print(f"Total: {total:,} | Pos (1): {n_pos:,} ({100*n_pos/total:.5f}%) | Neg (0): {n_neg:,}")
    print(f"scale_pos_weight sugerido: {ratio:,.0f}")
    return {'n_pos': n_pos, 'n_neg': n_neg, 'scale_pos_weight': ratio}


def build_resampler(y_train, strategy='auto', minority_target_ratio=0.05):
    """
    Monta resampler seguro para desbalanceamento extremo.

    - 'none'     : sem resampling (usa só class weights nos modelos)
    - 'under'    : undersampling da majoritária
    - 'smote'    : undersample + SMOTE (se n_pos >= 6)
    - 'auto'     : escolhe automaticamente conforme n_pos
    """
    n_pos = int((np.asarray(y_train) == 1).sum())

    if strategy == 'auto':
        strategy = 'smote' if n_pos >= 6 else ('under' if n_pos >= 2 else 'none')

    if strategy == 'none':
        return None, strategy

    if strategy == 'under':
        # mantém todas as positivas; reduz negativas para ratio alvo
        rus = RandomUnderSampler(
            sampling_strategy=minority_target_ratio,
            random_state=RANDOM_STATE
        )
        return rus, strategy

    if strategy == 'smote':
        # 1º undersample para ratio intermediário, depois SMOTE
        k_neighbors = min(5, n_pos - 1)
        smote = SMOTE(
            sampling_strategy='auto',
            k_neighbors=k_neighbors,
            random_state=RANDOM_STATE
        )
        rus = RandomUnderSampler(
            sampling_strategy=min(minority_target_ratio, 0.10),
            random_state=RANDOM_STATE
        )
        # pipeline manual: under -> smote
        return ('under_smote', rus, smote), strategy

    raise ValueError(f"strategy desconhecida: {strategy}")


def apply_resampler(resampler, X, y):
    if resampler is None:
        return X, y
    if isinstance(resampler, tuple) and resampler[0] == 'under_smote':
        _, rus, smote = resampler
        X_u, y_u = rus.fit_resample(X, y)
        return smote.fit_resample(X_u, y_u)
    return resampler.fit_resample(X, y)


def find_best_threshold(y_true, y_proba, metric='f1', min_precision=None):
    """Otimiza threshold na curva Precision-Recall."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    thresholds = np.append(thresholds, 1.0)

    best_t, best_score = 0.5, -1
    for p, r, t in zip(precisions, recalls, thresholds):
        if min_precision and p < min_precision:
            continue
        score = (2 * p * r / (p + r)) if metric == 'f1' and (p + r) > 0 else r
        if score > best_score:
            best_score, best_t = score, t
    return best_t, best_score

## 3. Factory de Modelos (já configurados para imbalance)

Famílias incluídas:
- **Boosting / árvores:** LGBM, XGB, CatBoost, HistGBM, RF, ExtraTrees, GBC, AdaBoost, BalancedRF, BalancedBagging
- **Lineares:** LogisticRegression, SGD, Ridge, PassiveAggressive, LinearSVC
- **Kernel (SVM):** SVC, NuSVC
- **Probabilísticos:** GaussianNB, BernoulliNB, MultinomialNB
- **Lazy / outros:** KNN, MLP, Dummy (baseline)

Use `include_groups` para filtrar: `'tree'`, `'linear'`, `'kernel'`, `'prob'`, `'lazy'`, `'baseline'`.

In [ ]:
def _normalize_models(models):
    """Aceita [(nome, modelo), ...] ou [modelo, ...]."""
    normalized = []
    for item in models:
        if isinstance(item, tuple):
            normalized.append(item)
        else:
            normalized.append((type(item).__name__, item))
    return normalized


def build_models(scale_pos_weight, include_groups=None):
    """
    Retorna lista de (nome, estimador) prontos para desbalanceamento extremo.

    include_groups: None (todos) ou subset de
        ['tree', 'linear', 'kernel', 'prob', 'lazy', 'baseline']
    """
    spw = float(scale_pos_weight)
    cw = 'balanced'

    catalog = {
        # --- boosting / árvores ---
        'tree': [
            ('LGBM', LGBMClassifier(
                random_state=RANDOM_STATE, n_jobs=-1,
                scale_pos_weight=spw, n_estimators=500, learning_rate=0.05,
            )),
            ('XGBoost', XGBClassifier(
                random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
                scale_pos_weight=spw, n_estimators=500, learning_rate=0.05,
                eval_metric='aucpr',
            )),
            ('CatBoost', CatBoostClassifier(
                random_state=RANDOM_STATE, verbose=False,
                auto_class_weights='Balanced', iterations=500, learning_rate=0.05,
            )),
            ('HistGBM', en.HistGradientBoostingClassifier(
                random_state=RANDOM_STATE, max_iter=500, learning_rate=0.05,
            )),
            ('RandomForest', en.RandomForestClassifier(
                random_state=RANDOM_STATE, n_jobs=-1,
                n_estimators=400, class_weight=cw,
            )),
            ('ExtraTrees', en.ExtraTreesClassifier(
                random_state=RANDOM_STATE, n_jobs=-1,
                n_estimators=400, class_weight=cw,
            )),
            ('GradientBoosting', en.GradientBoostingClassifier(
                random_state=RANDOM_STATE, n_estimators=300, learning_rate=0.05,
            )),
            ('AdaBoost', en.AdaBoostClassifier(
                random_state=RANDOM_STATE, n_estimators=300, learning_rate=0.05,
            )),
            ('BalancedRF', BalancedRandomForestClassifier(
                random_state=RANDOM_STATE, n_jobs=-1,
                n_estimators=300, sampling_strategy='auto',
            )),
            ('BalancedBagging', BalancedBaggingClassifier(
                estimator=DecisionTreeClassifier(max_depth=10, class_weight=cw),
                random_state=RANDOM_STATE, n_jobs=-1, n_estimators=100,
            )),
        ],

        # --- lineares ---
        'linear': [
            ('LogReg', lm.LogisticRegression(
                random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1,
                class_weight=cw, solver='saga', penalty='l2',
            )),
            ('SGD', lm.SGDClassifier(
                random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1,
                class_weight=cw, loss='log_loss', penalty='l2', alpha=1e-4,
            )),
            ('Ridge', lm.RidgeClassifier(
                random_state=RANDOM_STATE, class_weight=cw, alpha=1.0,
            )),
            ('PassiveAggressive', lm.PassiveAggressiveClassifier(
                random_state=RANDOM_STATE, max_iter=1000, class_weight=cw,
            )),
            ('LinearSVC', LinearSVC(
                random_state=RANDOM_STATE, max_iter=3000, class_weight=cw,
            )),
        ],

        # --- kernel ---
        'kernel': [
            ('SVC_rbf', SVC(
                random_state=RANDOM_STATE, kernel='rbf',
                class_weight=cw, probability=False,
            )),
            ('SVC_linear', SVC(
                random_state=RANDOM_STATE, kernel='linear',
                class_weight=cw, probability=False,
            )),
            ('NuSVC', NuSVC(
                random_state=RANDOM_STATE, class_weight=cw, probability=False,
            )),
        ],

        # --- probabilísticos ---
        'prob': [
            ('GaussianNB', GaussianNB()),
            ('BernoulliNB', BernoulliNB()),      # ideal p/ features binárias ou >= 0
            ('MultinomialNB', MultinomialNB()),  # ideal p/ features >= 0
        ],

        # --- lazy / neural ---
        'lazy': [
            ('KNN', nh.KNeighborsClassifier(
                n_neighbors=7, weights='distance', n_jobs=-1,
            )),
            ('MLP', MLPClassifier(
                random_state=RANDOM_STATE, max_iter=500,
                hidden_layer_sizes=(128, 64), early_stopping=True,
            )),
        ],

        # --- baseline ---
        'baseline': [
            ('Dummy_stratified', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE)),
            ('Dummy_prior', DummyClassifier(strategy='prior', random_state=RANDOM_STATE)),
        ],
    }

    if include_groups is None:
        include_groups = list(catalog.keys())

    models = []
    for group in include_groups:
        models.extend(catalog[group])

    return models

## 4. ML Training Loop (PRAUC)

In [ ]:
def ml_training_loop(
    models, X_train, y_train, X_val, y_val,
    resampler=None, sort=True, verbose=True, tune_threshold=True
):
    """
    Treina lista de modelos e avalia no hold-out.
    Retorna DataFrame de métricas (ordenável por PRAUC) + dict de artefatos.
    """
    models = _normalize_models(models)
    if sort:
        models = sorted(models, key=lambda x: x[0])

    X_fit, y_fit = apply_resampler(resampler, X_train, y_train)
    if verbose and resampler is not None:
        print(f"Após resampling: {len(y_fit):,} amostras | pos={ (y_fit==1).sum():, }")

    rows = []
    artifacts = {}

    for i, (name, model) in enumerate(models, 1):
        if verbose:
            print(f"[{i}/{len(models)}] Treinando {name}...")

        t0 = time()
        try:
            fitted = model.fit(X_fit, y_fit)
        except Exception as e:
            print(f"  ERRO em {name}: {e}")
            continue
        train_sec = time() - t0

        y_proba = _get_proba(fitted, X_val)
        threshold = 0.5
        if tune_threshold:
            threshold, _ = find_best_threshold(y_val, y_proba, metric='f1')

        y_pred = (y_proba >= threshold).astype(int)
        metrics = metrics_from_ml_loop(y_val, y_pred, y_proba)
        metrics['Model_Name'] = name
        metrics['Threshold']  = threshold
        metrics['Train_sec']  = round(train_sec, 2)
        rows.append(metrics)

        artifacts[name] = {
            'model'     : fitted,
            'threshold' : threshold,
            'metrics'   : metrics,
            'resampler' : resampler,
        }

    results = pd.DataFrame(rows).set_index('Model_Name')
    col_order = ['PRAUC', 'ROC-AUC', 'F1', 'Precision', 'Recall', 'Accuracy', 'Threshold', 'Train_sec']
    results = results[[c for c in col_order if c in results.columns]]
    return results.sort_values('PRAUC', ascending=False), artifacts

## 5. Cross Validation (Stratified, PRAUC)

In [ ]:
def cross_validation(models, X_train, y_train, k=5, resampler=None, sort=True):
    models = _normalize_models(models)
    if sort:
        models = sorted(models, key=lambda x: x[0])

    kfold = ms.StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    rows = []

    for j, (name, model) in enumerate(models, 1):
        print(f"CV [{j}/{len(models)}] {name}")
        fold_scores = {m: [] for m in ['PRAUC', 'F1', 'Recall', 'Precision']}

        for fold, (tr_idx, val_idx) in enumerate(kfold.split(X_train, y_train), 1):
            X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
            X_va, y_va = X_train[val_idx], y_train[val_idx]

            X_tr, y_tr = apply_resampler(resampler, X_tr, y_tr)
            try:
                fitted = model.fit(X_tr, y_tr)
            except Exception as e:
                print(f"  ERRO fold {fold} em {name}: {e}")
                continue

            y_proba = _get_proba(fitted, X_va)
            thr, _ = find_best_threshold(y_va, y_proba)
            y_pred = (y_proba >= thr).astype(int)
            m = metrics_from_ml_loop(y_va, y_pred, y_proba)

            for key in fold_scores:
                fold_scores[key].append(m[key])

        row = {'Model_Name': name}
        for key, vals in fold_scores.items():
            if vals:
                row[key] = f"{np.mean(vals):.4f} +/- {np.std(vals):.4f}"
            else:
                row[key] = 'N/A'
        rows.append(row)

    return pd.DataFrame(rows).set_index('Model_Name')

## 6. Persistência (salvar / carregar para produção)

In [ ]:
def save_model_artifact(artifact, name, directory=MODELS_DIR):
    """Salva modelo + threshold + metadados."""
    directory = Path(directory)
    directory.mkdir(exist_ok=True)

    payload = {
        'model'      : artifact['model'],
        'threshold'  : artifact['threshold'],
        'metrics'    : artifact['metrics'],
        'saved_at'   : dt.now().isoformat(),
        'primary_metric': PRIMARY_METRIC,
    }

    path = directory / f"{name}.pkl"
    with open(path, 'wb') as f:
        pickle.dump(payload, f)

    # metadados legíveis
    meta_path = directory / f"{name}_meta.json"
    meta = {k: v for k, v in payload.items() if k != 'model'}
    meta['metrics'] = {k: float(v) if isinstance(v, (np.floating, float)) else v
                       for k, v in meta['metrics'].items()}
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    print(f"Salvo: {path}")
    return path


def load_model_artifact(name, directory=MODELS_DIR):
    path = Path(directory) / f"{name}.pkl"
    with open(path, 'rb') as f:
        return pickle.load(f)

## 7. Wrapper de Produção

Interface unificada: recebe `X` (obrigatório) e `y` (opcional).

In [ ]:
class ProductionPredictor:
    """Carrega artefato salvo e expõe predict / predict_proba / evaluate."""

    def __init__(self, artifact_path=None, artifact=None):
        if artifact is None:
            with open(artifact_path, 'rb') as f:
                artifact = pickle.load(f)
        self.model     = artifact['model']
        self.threshold = artifact.get('threshold', 0.5)
        self.metrics   = artifact.get('metrics', {})

    def predict_proba(self, X):
        return _get_proba(self.model, X)

    def predict(self, X, threshold=None):
        thr = self.threshold if threshold is None else threshold
        return (self.predict_proba(X) >= thr).astype(int)

    def evaluate(self, X, y):
        """Só quando y estiver disponível (validação offline)."""
        y_proba = self.predict_proba(X)
        y_pred  = self.predict(X)
        return metrics_from_ml_loop(y, y_pred, y_proba)

---
# TREINO — Snippets

A partir daqui, rode célula a célula.  
Pressupõe que você já tem `X_train`, `X_test`, `y_train`, `y_test` prontos.

### 8.1 Diagnóstico do desbalanceamento

In [ ]:
# --- seus dados ---
# X_train, X_test, y_train, y_test = ...

print('=== TRAIN ===')
imb_train = imbalance_report(y_train)
print('\n=== TEST ===')
imb_test = imbalance_report(y_test)

### 8.2 Configurar resampling & modelos

In [ ]:
RESAMPLING_STRATEGY = 'auto'   # 'none' | 'under' | 'smote' | 'auto'
MINORITY_TARGET_RATIO = 0.05   # após undersample: 5% positivos

resampler, strategy_used = build_resampler(
    y_train,
    strategy=RESAMPLING_STRATEGY,
    minority_target_ratio=MINORITY_TARGET_RATIO,
)
print(f"Resampling: {strategy_used}")

# None = todos os 25 modelos | ou filtre: ['tree', 'linear', 'kernel', 'prob', 'lazy', 'baseline']
INCLUDE_GROUPS = None

models = build_models(
    scale_pos_weight=imb_train['scale_pos_weight'],
    include_groups=INCLUDE_GROUPS,
)
print(f"Total de modelos: {len(models)}")

### 8.3 Rodar training loop

In [ ]:
ml_results, trained_artifacts = ml_training_loop(
    models,
    X_train, y_train,
    X_test,  y_test,       # hold-out de validação
    resampler=resampler,
    verbose=True,
    tune_threshold=True,
)

ml_results.style.background_gradient(subset=['PRAUC'], cmap='YlGn').format('{:.4f}')

### 8.4 (Opcional) Cross Validation

In [ ]:
DO_CV = False

if DO_CV:
    cv_results = cross_validation(
        models, X_train, y_train,
        k=5, resampler=resampler
    )
    display(cv_results)

### 8.5 Salvar modelos escolhidos

In [ ]:
# salvar o melhor por PRAUC
best_name = ml_results.index[0]
save_model_artifact(trained_artifacts[best_name], name=f"best_{best_name}")

# --- ou salvar manualmente os que quiser ---
MODELS_TO_SAVE = ['LGBM', 'XGBoost', 'LogReg', 'LinearSVC', 'GaussianNB']

for model_name in MODELS_TO_SAVE:
    if model_name in trained_artifacts:
        save_model_artifact(trained_artifacts[model_name], name=model_name)

### 8.6 (Opcional) Fine-tuning com BayesSearchCV — otimiza PRAUC

In [ ]:
DO_FINE_TUNING = False
TARGET_MODEL   = 'LGBM'  # 'LGBM' | 'XGBoost' | 'LogReg' | 'RandomForest' | ...

if DO_FINE_TUNING:
    spw = imb_train['scale_pos_weight']

    if TARGET_MODEL == 'LGBM':
        estimator = LGBMClassifier(
            random_state=RANDOM_STATE, n_jobs=-1,
            scale_pos_weight=spw,
        )
        param_space = {
            'n_estimators'    : (200, 800),
            'learning_rate'   : (1e-3, 0.2, 'log-uniform'),
            'num_leaves'      : (20, 120),
            'min_child_samples': (1, 20),
            'subsample'       : (0.6, 1.0),
            'colsample_bytree': (0.6, 1.0),
        }
    elif TARGET_MODEL == 'XGBoost':
        estimator = XGBClassifier(
            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
            scale_pos_weight=spw, eval_metric='aucpr',
        )
        param_space = {
            'n_estimators'    : (200, 800),
            'learning_rate'   : (1e-3, 0.2, 'log-uniform'),
            'max_depth'       : (3, 12),
            'subsample'       : (0.6, 1.0),
            'colsample_bytree': (0.6, 1.0),
        }
    else:
        raise ValueError(TARGET_MODEL)

    X_fit, y_fit = apply_resampler(resampler, X_train, y_train)

    t0 = time()
    print(f"Fine-tuning {TARGET_MODEL} iniciado: {dt.now()}")

    bayes_search = BayesSearchCV(
        estimator,
        param_space,
        scoring=PRAUC_SCORER,
        n_iter=32,
        cv=ms.StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    bayes_search.fit(X_fit, y_fit)

    best_tuned = bayes_search.best_estimator_
    y_proba = _get_proba(best_tuned, X_test)
    thr, _ = find_best_threshold(y_test, y_proba)
    y_pred = (y_proba >= thr).astype(int)

    tuned_metrics = metrics_from_ml_loop(y_test, y_pred, y_proba)
    print(f"Concluído em {(time()-t0)/60:.1f} min | PRAUC={tuned_metrics['PRAUC']:.4f}")
    print(f"Best params: {bayes_search.best_params_}")

    tuned_artifact = {
        'model'    : best_tuned,
        'threshold': thr,
        'metrics'  : tuned_metrics,
    }
    save_model_artifact(tuned_artifact, name=f"tuned_{TARGET_MODEL}")

### 8.7 (Opcional) Optuna — também otimiza PRAUC

In [ ]:
DO_OPTUNA = False

if DO_OPTUNA:
    spw = imb_train['scale_pos_weight']
    X_fit, y_fit = apply_resampler(resampler, X_train, y_train)
    kfold = ms.StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial):
        params = {
            'n_estimators'     : trial.suggest_int('n_estimators', 200, 800, step=50),
            'learning_rate'    : trial.suggest_float('learning_rate', 1e-3, 0.2, log=True),
            'num_leaves'       : trial.suggest_int('num_leaves', 20, 120),
            'min_child_samples': trial.suggest_int('min_child_samples', 1, 20),
            'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'scale_pos_weight' : spw,
            'random_state'     : RANDOM_STATE,
            'n_jobs'           : -1,
        }
        model = LGBMClassifier(**params)
        scores = ms.cross_val_score(
            model, X_fit, y_fit, cv=kfold, scoring=PRAUC_SCORER, n_jobs=-1
        )
        return scores.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)

    print(f"Best PRAUC (CV): {study.best_value:.4f}")
    print(study.best_params)

---
# PRODUÇÃO — Snippets

Carrega modelo salvo e faz inferência.  
`y` é opcional — use apenas quando quiser validar offline.

### 9.1 Carregar modelo

In [ ]:
MODEL_PATH = MODELS_DIR / 'best_LGBM.pkl'  # ajuste o nome

predictor = ProductionPredictor(artifact_path=MODEL_PATH)
print(f"Threshold: {predictor.threshold:.4f}")
print(f"PRAUC (treino): {predictor.metrics.get('PRAUC', 'N/A')}")

### 9.2 Inferência — somente X (caso padrão)

In [ ]:
# X = ...  # features de produção (mesmo formato de X_train)

y_proba = predictor.predict_proba(X)
y_pred  = predictor.predict(X)

# resultado
df_pred = pd.DataFrame({'proba': y_proba, 'pred': y_pred})
df_pred.head()

### 9.3 Inferência + validação offline (X + y)

In [ ]:
# X, y = ...  # y só quando disponível

prod_metrics = predictor.evaluate(X, y)
print(f"PRAUC produção: {prod_metrics['PRAUC']:.4f}")
print(f"F1: {prod_metrics['F1']:.4f} | Recall: {prod_metrics['Recall']:.4f}")

y_pred = predictor.predict(X)
print(classification_report(y, y_pred, zero_division=0))
print('Confusion Matrix:\n', confusion_matrix(y, y_pred))

### 9.4 Uso em script/serviço (fora do notebook)

In [ ]:
# Exemplo mínimo para deploy:
#
# from pathlib import Path
# import pickle
# import numpy as np
#
# def load_predictor(path='./models/best_LGBMClassifier.pkl'):
#     with open(path, 'rb') as f:
#         art = pickle.load(f)
#     return art['model'], art['threshold']
#
# model, threshold = load_predictor()
#
# def predict_batch(X):
#     proba = model.predict_proba(X)[:, 1]
#     return (proba >= threshold).astype(int), proba